# k-Means Clustering mit italienischen Weinen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_20/wine_clustering_kmeans.ipynb)

In diesem Notebook lernen wir K-Means praktisch kennen. Wir verwenden italienische Weine mit verschiedenen chemischen Eigenschaften und vergleichen die gefundenen Cluster mit drei bekannten Rebsorten-Klassen.

Der Wine-Datensatz hat 13 chemische Eigenschaften als Features. Wir verwenden zunächst nur zwei davon und machen mit diesen beiden das Clustering. Am Ende des Notebooks berechnen wir dann ein weiteres Clustering auf allen 13 Features zum Vergleich.

**Ziel:** Verstehen, wie K-Means funktioniert und wie die Skalierung die Gruppierung beeinflusst.

Die Klassen stehen für drei Rebsorten, nicht für Produzenten. Quelle: [Wine-Datensatz der UCI](https://archive.ics.uci.edu/dataset/109/wine).

## 1. Bibliotheken importieren und Daten laden

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score

# Wein-Daten laden
wine = load_wine()
print("Wine-Dataset geladen!")
print(f"Anzahl Weine: {wine.data.shape[0]}")
print(f"Anzahl chemische Eigenschaften: {wine.data.shape[1]}")
print(f"Rebsorten-Klassen: {wine.target_names}")
print(f"\nErste 5 Eigenschaften: {wine.feature_names[:5]}")

## 2. Das Skalierungs-Problem verstehen

Wir schauen uns zwei Eigenschaften mit **völlig verschiedenen Skalen** an:

In [ ]:
# Zwei Features mit sehr verschiedenen Skalen wählen
X = wine.data[:, [0, 12]]  # Alkohol und Proline
feature_names = ['Alkohol (%)', 'Proline (Datensatzwerte)']

print("🍷 Wertebereich der chemischen Eigenschaften:")
print(f"Alkohol: {X[:, 0].min():.1f} - {X[:, 0].max():.1f} %")
print(f"Proline (Datensatzwerte): {X[:, 1].min():.0f} - {X[:, 1].max():.0f}")
print("\n⚠️ RIESIGE Unterschiede in den Wertebereichen!")
print("Proline hat einen viel größeren Zahlenbereich als Alkohol.")
print("Ohne Skalierung beeinflusst Proline die Abstände viel stärker als Alkohol.")

## 3. Originaldaten visualisieren

Zuerst schauen wir uns die echten Rebsorten-Klassen an. Die Farben zeigen diese bekannten Klassen; sie werden später nicht als Eingabe für K-Means verwendet.

In [ ]:
# Originaldaten mit echten Rebsorten-Klassen plotten
plt.figure(figsize=(12, 5))

# Plot 1: Normale Skala
plt.subplot(1, 2, 1)
colors = ['red', 'blue', 'green']
class_names = ['Rebsorte A', 'Rebsorte B', 'Rebsorte C']
for i, class_name in enumerate(class_names):
    mask = wine.target == i
    plt.scatter(X[mask, 0], X[mask, 1], c=colors[i], label=class_name, alpha=0.7, s=50)

plt.xlabel('Alkohol (%)')
plt.ylabel('Proline (Datensatzwerte)')
plt.title('Wein-Daten: Echte Rebsorten-Klassen')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Log-Skala um das Problem zu zeigen
plt.subplot(1, 2, 2)
for i, class_name in enumerate(class_names):
    mask = wine.target == i
    plt.scatter(X[mask, 0], X[mask, 1], c=colors[i], label=class_name, alpha=0.7, s=50)

plt.xlabel('Alkohol (%)')
plt.ylabel('Proline (Datensatzwerte)')
plt.yscale('log')  # Log-Skala für Proline
plt.title('Mit Log-Skala (bessere Sicht)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Wir sehen drei relativ gut getrennte Gruppen.")
print("Kann k-Means diese automatisch finden?")
print("Problem: Die y-Achse ist viel größer als die x-Achse!")

## 4. Daten skalieren

**Kritisch wichtig:** k-Means ist sehr empfindlich gegenüber unterschiedlichen Wertebereichen!

In [ ]:
# Daten skalieren (Mittelwert 0, Standardabweichung 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Nach der Skalierung:")
print(f"Alkohol: Mittelwert={X_scaled[:, 0].mean():.3f}, Std={X_scaled[:, 0].std():.3f}")
print(f"Proline: Mittelwert={X_scaled[:, 1].mean():.3f}, Std={X_scaled[:, 1].std():.3f}")
print("\n🎯 Beide Features haben jetzt dieselbe Standardabweichung, nicht denselben Wertebereich.")
print("Beide haben Mittelwert 0 und Standardabweichung 1.")

# Skalierte Daten visualisieren
plt.figure(figsize=(10, 6))
for i, class_name in enumerate(class_names):
    mask = wine.target == i
    plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1], c=colors[i], label=class_name, alpha=0.7, s=50)

plt.xlabel('Alkohol (skaliert)')
plt.ylabel('Proline (skaliert)')
plt.title('Wein-Daten: Skaliert')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Proline dominiert die Abstände jetzt nicht mehr allein wegen seiner größeren Streuung.")

## 5. k-Means mit k=3 anwenden

Für dieses Lernbeispiel wählen wir k=3, passend zu den drei bekannten Rebsorten-Klassen. K-Means bekommt diese Klassenlabels nicht als Eingabe und muss nicht genau dieselben Gruppen finden.

In [ ]:
# k-Means mit k=3 anwenden
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
centers = kmeans.cluster_centers_

print("🤖 k-Means Ergebnisse:")
print("Cluster-Zentren (skaliert):")
for i, center in enumerate(centers):
    print(f"  Cluster {i}: [{center[0]:.2f}, {center[1]:.2f}]")

# Wie viele Weine in jedem Cluster?
unique, counts = np.unique(cluster_labels, return_counts=True)
print("\n📊 Anzahl Weine pro Cluster:")
for cluster, count in zip(unique, counts):
    print(f"  Cluster {cluster}: {count} Weine")

print("\n🎯 Zur Erinnerung - echte Verteilung:")
unique_real, counts_real = np.unique(wine.target, return_counts=True)
for class_name, count in zip(class_names, counts_real):
    print(f"  {class_name}: {count} Weine")

## 6. Ergebnisse visualisieren

Vergleichen wir die echten Rebsorten-Klassen mit den K-Means-Clustern. Clusternummern sind beliebige Namen; gleiche Farben in verschiedenen Diagrammen müssen nicht dieselbe Gruppe bedeuten.

In [ ]:
# Drei Plots nebeneinander
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Echte Rebsorten-Klassen (Original-Skala)
for i, class_name in enumerate(class_names):
    mask = wine.target == i
    axes[0].scatter(X[mask, 0], X[mask, 1], c=colors[i], label=class_name, alpha=0.7, s=50)
axes[0].set_xlabel('Alkohol (%)')
axes[0].set_ylabel('Proline (Datensatzwerte)')
axes[0].set_title('Echte Rebsorten-Klassen')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: k-Means Cluster (Original-Skala für besseres Verständnis)
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7, s=50)
# Zentren zurück-transformieren für bessere Anschauung
centers_orig = scaler.inverse_transform(centers)
axes[1].scatter(centers_orig[:, 0], centers_orig[:, 1], c='red', 
               marker='x', s=200, linewidths=3, label='Cluster-Zentren')
axes[1].set_xlabel('Alkohol (%)')
axes[1].set_ylabel('Proline (Datensatzwerte)')
axes[1].set_title('k-Means Cluster (k=3)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1], label='Cluster')

# Plot 3: Skalierte Daten mit Clustern
scatter2 = axes[2].scatter(X_scaled[:, 0], X_scaled[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7, s=50)
axes[2].scatter(centers[:, 0], centers[:, 1], c='red', 
               marker='x', s=200, linewidths=3, label='Cluster-Zentren')
axes[2].set_xlabel('Alkohol (skaliert)')
axes[2].set_ylabel('Proline (skaliert)')
axes[2].set_title('k-Means auf skalierten Daten')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=axes[2], label='Cluster')

plt.tight_layout()
plt.show()

print("👀 Beobachtungen:")
print("• Links: Wie die Daten wirklich aussehen")
print("• Mitte: Was k-Means gefunden hat (auf normale Skala zurück projiziert)")
print("• Rechts: Wie k-Means die Daten 'sieht' (skaliert)")

## 7. Ergebnisse bewerten

Wir prüfen zwei unterschiedliche Fragen:
- **ARI:** Wie gut stimmen die Cluster mit den bekannten Rebsorten-Klassen überein? 1 bedeutet perfekte Übereinstimmung, etwa 0 eine zufällige Zuordnung.
- **Silhouette-Score:** Passen die Punkte nach ihren Abständen gut zu den gefundenen Clustern? Dafür brauchen wir keine echten Labels.

Für jeden Punkt werden zwei mittlere Abstände verglichen: zu den anderen Punkten seines Clusters und zu den Punkten des nächstgelegenen anderen Clusters. Der Score liegt zwischen −1 und 1:
- **Nahe 1:** Der Punkt liegt deutlich näher an seinem eigenen Cluster.
- **Nahe 0:** Die Zuordnung ist nach den Abständen wenig eindeutig.
- **Negativ:** Der Punkt liegt im Durchschnitt näher an einem anderen Cluster.

Der Gesamtscore ist der Durchschnitt über alle Punkte. Er bewertet die Trennung nach Abständen, nicht die Übereinstimmung mit Rebsorten.

In [ ]:
# Bewertungsmetriken berechnen
ari = adjusted_rand_score(wine.target, cluster_labels)
sil_score = silhouette_score(X_scaled, cluster_labels)

print("📊 Bewertung der Cluster-Qualität:")
print(f"Adjusted Rand Index (ARI): {ari:.3f}")
print(f"Silhouette Score: {sil_score:.3f}")
print()
print("🔍 Interpretation:")
print(f"• ARI = {ari:.3f}: Übereinstimmung mit den Rebsorten-Klassen (1 = perfekt).")
print(f"• Silhouette = {sil_score:.3f}: Trennung und Kompaktheit auf den skalierten Daten.")

## 8. Experiment: Warum Skalierung so wichtig ist

Schauen wir, was passiert, wenn wir die Daten **nicht** skalieren.

**Was lernen wir aus dem Silhouette-Score?** Ohne Skalierung dominiert Proline die Abstände. Gruppen nach niedrigen, mittleren und hohen Proline-Werten können dadurch gut getrennt wirken. Ein hoher Score bedeutet aber nicht automatisch, dass diese Gruppen den Rebsorten entsprechen.

**Wichtig:** Skalierte und unskalierte Daten verwenden unterschiedliche Abstände. Deshalb sind ihre Silhouette-Scores nicht direkt als besser oder schlechter vergleichbar. Das gilt auch für den späteren Vergleich von 2 und 13 Merkmalen.

**Sinnvoller Einsatz:** Halte Merkmale und Skalierung gleich und vergleiche zum Beispiel k=2, k=3 und k=4 oder verschiedene Initialisierungen. Dann hilft der Score, kompaktere und besser getrennte Aufteilungen zu erkennen. Ein Beispiel findest du im [Elbow-Notebook](../Vorlesung_19/wine_elbow_method.ipynb).

**Merksatz:** Eine nach Abständen gut getrennte Gruppierung muss nicht die Gruppierung sein, die uns fachlich interessiert.

In [ ]:
# k-Means OHNE Skalierung
print("🧪 Experiment: k-Means ohne Skalierung")
print("Erinnerung: Alkohol etwa 11-15%, Proline-Datensatzwerte etwa 278-1680")

kmeans_unscaled = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels_unscaled = kmeans_unscaled.fit_predict(X)

# Vergleich visualisieren
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Mit Skalierung
scatter1 = axes[0].scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7, s=50)
centers_orig = scaler.inverse_transform(centers)
axes[0].scatter(centers_orig[:, 0], centers_orig[:, 1], c='red', 
               marker='x', s=200, linewidths=3, label='Zentren')
axes[0].set_xlabel('Alkohol (%)')
axes[0].set_ylabel('Proline (Datensatzwerte)')
axes[0].set_title('MIT Skalierung')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0])

# Ohne Skalierung
scatter2 = axes[1].scatter(X[:, 0], X[:, 1], c=cluster_labels_unscaled, cmap='viridis', alpha=0.7, s=50)
axes[1].scatter(kmeans_unscaled.cluster_centers_[:, 0], kmeans_unscaled.cluster_centers_[:, 1], 
               c='red', marker='x', s=200, linewidths=3, label='Zentren')
axes[1].set_xlabel('Alkohol (%)')
axes[1].set_ylabel('Proline (Datensatzwerte)')
axes[1].set_title('OHNE Skalierung')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.show()

# Metriken vergleichen
ari_unscaled = adjusted_rand_score(wine.target, cluster_labels_unscaled)
sil_unscaled = silhouette_score(X, cluster_labels_unscaled)

print("\n📊 Vergleich der Metriken:")
print(f"{'Methode':<20} {'ARI':<8} {'Silhouette':<12}")
print("-" * 40)
print(f"{'Mit Skalierung':<20} {ari:.3f}{'':>4} {sil_score:.3f}")
print(f"{'Ohne Skalierung':<20} {ari_unscaled:.3f}{'':>4} {sil_unscaled:.3f}")

print("\n💡 Interpretation:")
print("• Ohne Skalierung bewertet Silhouette hauptsächlich die Trennung nach Proline.")
print("• Die beiden Silhouette-Scores verwenden unterschiedliche Abstände; vergleicht sie nicht direkt.")
print(f"• ARI mit Skalierung ({ari:.3f}) vs. ohne ({ari_unscaled:.3f}): Vergleich mit denselben Referenzklassen.")
print("\n💡 Lernerfahrung:")
print("• Verschiedene Metriken können verschiedene Geschichten erzählen")
print("• Ohne Skalierung hängt die Gewichtung stark von den Einheiten und der Streuung ab.")
print("• StandardScaler verändert diese Gewichtung. Hier wollen wir den Größeneffekt von Proline vermeiden.")

## 9. Bonus: k-Means mit allen 13 Features

Bisher haben wir nur 2 Features verwendet (Alkohol und Proline). Aber der Wine-Datensatz hat **13 chemische Eigenschaften**! Was passiert, wenn wir alle nutzen?

**Hypothese:** Die zusätzlichen Merkmale könnten helfen, die Rebsorten-Klassen besser wiederzufinden. Mehr Merkmale helfen aber nicht automatisch: Unwichtige Merkmale können die Gruppierung auch erschweren.

In [ ]:
# k-Means mit ALLEN 13 Features
print("🚀 k-Means mit allen 13 chemischen Eigenschaften:")
print(f"Features: {wine.feature_names}")
print()

# Alle Daten skalieren
X_all = wine.data  # Alle 13 Features
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

print("📊 Daten-Info:")
print(f"   • Anzahl Features: {X_all.shape[1]}")
print(f"   • Anzahl Weine: {X_all.shape[0]}")
print(f"   • Nach Skalierung: Mittelwerte ≈ {X_all_scaled.mean():.3f}, Std ≈ {X_all_scaled.std():.3f}")

# k-Means mit allen Features anwenden
kmeans_all = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels_all = kmeans_all.fit_predict(X_all_scaled)

# Metriken berechnen
ari_all = adjusted_rand_score(wine.target, cluster_labels_all)
sil_all = silhouette_score(X_all_scaled, cluster_labels_all)

print("\n🎯 Ergebnisse mit allen 13 Features:")
print(f"   • ARI: {ari_all:.3f}")
print(f"   • Silhouette Score: {sil_all:.3f}")

In [ ]:
# Vergleich aller drei Ansätze visualisieren
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Echte Rebsorten-Klassen (als Referenz)
for i, class_name in enumerate(class_names):
    mask = wine.target == i
    axes[0].scatter(X[mask, 0], X[mask, 1], c=colors[i], label=class_name, alpha=0.7, s=50)
axes[0].set_xlabel('Alkohol (%)')
axes[0].set_ylabel('Proline (Datensatzwerte)')
axes[0].set_title('Echte Rebsorten-Klassen\n(Referenz)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. k-Means mit 2 Features (bisheriges Ergebnis)
scatter1 = axes[1].scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7, s=50)
axes[1].set_xlabel('Alkohol (%)')
axes[1].set_ylabel('Proline (Datensatzwerte)')
axes[1].set_title(f'k-Means (2 Features)\nARI: {ari:.3f}, Sil: {sil_score:.3f}')
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[1], label='Cluster')

# 3. k-Means mit allen 13 Features (projiziert auf Alkohol/Proline für Visualisierung)
scatter2 = axes[2].scatter(X[:, 0], X[:, 1], c=cluster_labels_all, cmap='viridis', alpha=0.7, s=50)
axes[2].set_xlabel('Alkohol (%)')
axes[2].set_ylabel('Proline (Datensatzwerte)')
axes[2].set_title(f'k-Means (alle 13 Features)\nARI: {ari_all:.3f}, Sil: {sil_all:.3f}')
axes[2].grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=axes[2], label='Cluster')

plt.tight_layout()
plt.show()

print("👀 Beobachtung:")
print("   Obwohl wir nur Alkohol vs. Proline sehen, nutzt der rechte Plot")
print("   alle 13 Features für das Clustering. Die Farben zeigen das Ergebnis!")

In [ ]:
# Detaillierter Vergleich der drei Ansätze
print("📊 Detaillierter Vergleich aller Ansätze:")
print()
print(f"{'Methode':<25} {'Features':<10} {'ARI':<8} {'Silhouette':<12} {'Bewertung'}")
print("-" * 70)

# ARI-Bewertung
def get_ari_rating(ari_val):
    if ari_val > 0.8: 
        return "Exzellent"
    elif ari_val > 0.6: 
        return "Sehr gut"
    elif ari_val > 0.4: 
        return "Gut"
    elif ari_val > 0.2: 
        return "Mäßig"
    else: 
        return "Schlecht"

print(f"{'Mit Skalierung (2D)':<25} {'2':<10} {ari:.3f}{'':>4} {sil_score:.3f}{'':>8} {get_ari_rating(ari)}")
print(f"{'Ohne Skalierung (2D)':<25} {'2':<10} {ari_unscaled:.3f}{'':>4} {sil_unscaled:.3f}{'':>8} {get_ari_rating(ari_unscaled)}")
print(f"{'Alle Features (13D)':<25} {'13':<10} {ari_all:.3f}{'':>4} {sil_all:.3f}{'':>8} {get_ari_rating(ari_all)}")

print("\n🎯 Erkenntnisse:")
change_2d = ari - ari_unscaled
change_13d = ari_all - ari

print(f"   • Skalierung (2D): Änderung des ARI um {change_2d:+.3f}")
print(f"   • Mehr Features (13D): Änderung des ARI um {change_13d:+.3f}")
print("   • Die Silhouette-Scores stammen aus verschiedenen Datenräumen und sind nicht direkt vergleichbar.")

print("\n💡 Was bedeutet das?")
if ari_all > ari:
    print("   ✅ In diesem Beispiel passen die Cluster mit 13 Merkmalen besser zu den Referenzklassen.")
    print("   ✅ k-Means kann alle 13 chemischen Eigenschaften nutzen")
    print("   ✅ Die zusätzlichen Features enthalten wertvolle Informationen")
else:
    print("   🤔 Mehr Features helfen nicht immer automatisch")
    print("   🤔 Manchmal reichen die wichtigsten Features aus")
    print("   🤔 Prüft, ob die zusätzlichen Merkmale zur gesuchten Gruppierung passen.")

print(f"\n🏆 Höchster ARI im Vergleich: {'13 Features' if ari_all > max(ari, ari_unscaled) else '2 Features (skaliert)' if ari > ari_unscaled else '2 Features (unskaliert)'}")

## 10. Zusammenfassung und Erkenntnisse

Was haben wir gelernt?

In [ ]:
print("🎓 Wichtige Erkenntnisse aus diesem Notebook:")
print()
print("✅ Was gut funktioniert hat:")
print("   • Wine-Dataset zeigt Skalierungsproblem perfekt")
print("   • K-Means findet Gruppen, die teilweise mit den Rebsorten-Klassen übereinstimmen.")
print("   • Skalierung macht einen messbaren Unterschied")
print()
print("⚠️ Was zu beachten ist:")
print("   • k-Means findet nicht immer exakt die echten Gruppen")
print("   • Verschiedene Features haben verschiedene Skalen")
print("   • Ohne Skalierung können Merkmale mit großer Streuung dominieren.")
print("   • k-Means funktioniert am besten bei runden, gleich großen Clustern")
print()
print("🔧 Praktische Tipps:")
print("   • Skalierung prüfen: Hier hilft StandardScaler, den Größeneffekt von Proline zu vermeiden.")
print("   • Verschiedene Features anschauen (nicht nur 2)")
print("   • Verschiedene random_state probieren")
print("   • Ergebnisse immer visualisieren und interpretieren")
print()
print("📈 Unsere Ergebnisse:")
print(f"   • ARI ohne Skalierung (2 Features): {ari_unscaled:.3f}")
print(f"   • ARI mit Skalierung (2 Features): {ari:.3f}")
print(f"   • ARI mit allen 13 Features: {ari_all:.3f}")
print(f"   • Änderung des ARI durch Skalierung: {ari - ari_unscaled:+.3f}")
print(f"   • Änderung des ARI durch mehr Features: {ari_all - ari:+.3f}")
print()
print("🏆 Wichtigste Lernerfahrung:")
print("   1️⃣ Skalierung beeinflusst, wie stark die Merkmale die Abstände bestimmen.")
print("   2️⃣ Mehr relevante Features können die Ergebnisse verbessern")
print("   3️⃣ Silhouette bewertet Abstände; ARI vergleicht mit bekannten Referenzgruppen.")